# 02 — Qwen3.5-4B — Zero-shot

Experimento **zero-shot** executado sobre o conjunto fixo de **215 autores de teste**.

Este notebook reutiliza os módulos de `src/` e preserva as condições do experimento:

- modelo: `Qwen/Qwen3.5-4B`;
- zero-shot;
- 30 tags por autor;
- máximo de 50 publicações por autor;
- `max_input_tokens = 8192`;
- `max_new_tokens = 1024`;
- `do_sample = True`;
- `temperature = 0.7`;
- `top_p = 0.8`;
- `top_k = 20`;
- `min_p = 0.0`;
- `repetition_penalty = 1.0`;
- `enable_thinking = False`;
- `seed = 42`;
- avaliação com SBERT `paraphrase-multilingual-mpnet-base-v2`;
- matching global greedy 1-para-1 com limiar 0,75.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np

current = Path.cwd().resolve()
candidates = [current, *current.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "src").exists()), current)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.llm.qwen_inference import (
    GenerationConfig,
    carregar_perfis_estruturados,
    executar_inferencia,
)
from src.evaluation.semantic_matching import (
    carregar_qrels,
    construir_vocabulario,
    encodar_com_sbert,
    construir_matriz_similaridade,
    matching_greedy_1_to_1,
    salvar_npz,
    salvar_csv_avaliacoes_gerais,
)
from src.evaluation.metrics import (
    carregar_perfis_por_documento,
    avaliar_autor,
    salvar_csv_metricas,
)

print(f"Raiz do projeto: {PROJECT_ROOT}")

## 1. Caminhos

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"
RESULT_DIR = PROJECT_ROOT / "results" / "zero_shot" / "02_qwen3_5_4b_zero_shot"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

PROFILES_FILE = DATA_DIR / "perfis_estruturados_qwen.json"
DOCUMENT_PROFILES_FILE = DATA_DIR / "perfis_documento_qwen.json"
QRELS_FILE = DATA_DIR / "ground_truth" / "LExR-prof-qrels_filtrado"
SPLIT_FILE = DATA_DIR / "splits" / "split_autores_seed42.json"

TAGS_FILE = RESULT_DIR / "tags_brutas.json"
RANKING_FILE = RESULT_DIR / "ranking_tags.json"
CHECKPOINT_FILE = RESULT_DIR / "checkpoint.json"
ERRORS_FILE = RESULT_DIR / "erros.json"
METRICS_FILE = RESULT_DIR / "metricas_por_autor.csv"
MATCHING_FILE = RESULT_DIR / "avaliacoes_gerais.csv"
SIM_DIR = RESULT_DIR / "sim_matrices"

for name, path in {
    "perfis_estruturados_qwen.json": PROFILES_FILE,
    "perfis_documento_qwen.json": DOCUMENT_PROFILES_FILE,
    "LExR-prof-qrels_filtrado": QRELS_FILE,
    "split_autores_seed42.json": SPLIT_FILE,
}.items():
    print(f"{name:36s} -> {'OK' if path.exists() else 'não encontrado'}")

## 2. Seleção do conjunto de teste

O split é realizado em nível de autor e esta execução utiliza exclusivamente os 215 pesquisadores do conjunto de teste.

In [ ]:
perfis = carregar_perfis_estruturados(PROFILES_FILE)

with SPLIT_FILE.open("r", encoding="utf-8") as f:
    split = json.load(f)

autores_teste = [str(a) for a in split["test"]]
perfis_teste = {a: perfis[a] for a in autores_teste if a in perfis}

print(f"Autores definidos no split de teste: {len(autores_teste)}")
print(f"Autores encontrados nos perfis:     {len(perfis_teste)}")

if len(autores_teste) != 215:
    raise ValueError(f"O split de teste deveria conter 215 autores, mas contém {len(autores_teste)}.")

faltantes = sorted(set(autores_teste) - set(perfis_teste))
if faltantes:
    raise ValueError(f"{len(faltantes)} autores do teste não foram encontrados nos perfis.")

## 3. Configuração do modelo

In [ ]:
config = GenerationConfig(
    model_name="Qwen/Qwen3.5-4B",
    mode="zero-shot",
    n_tags=30,
    max_publicacoes=50,
    batch_size=4,
    do_sample=True,
    temperature=0.7,
    top_p=0.8,
    top_k=20,
    min_p=0.0,
    enable_thinking=False,
    repetition_penalty=1.0,
    max_new_tokens=1024,
    max_input_tokens=8192,
    seed=42,
    checkpoint_every_authors=12,
)

config.validate()
config

## 4. Inferência zero-shot

In [ ]:
tags_brutas, rankings = executar_inferencia(
    perfis_teste,
    config,
    caminho_saida=TAGS_FILE,
    caminho_checkpoint=CHECKPOINT_FILE,
    caminho_ranking=RANKING_FILE,
    caminho_erros=ERRORS_FILE,
)

print(f"Autores processados: {len(tags_brutas)}")

## 5. Preparação da avaliação semântica

A mesma configuração de avaliação é utilizada em todas as abordagens. O SBERT é empregado apenas para estabelecer correspondências semânticas e calcular Coverage/Diversity, sem alterar a ordem originalmente produzida pelo modelo.

In [ ]:
from sentence_transformers import SentenceTransformer

SBERT_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
THRESHOLD = 0.75
THRESHOLD_COVERAGE = 0.75
TOP_K_MATCHING = 20

gt_norm, gt_original = carregar_qrels(QRELS_FILE)
perfis_doc = carregar_perfis_por_documento(DOCUMENT_PROFILES_FILE)

autores_alvo = set(autores_teste)
gt_norm = {a: v for a, v in gt_norm.items() if a in autores_alvo}
gt_original = {a: v for a, v in gt_original.items() if a in autores_alvo}
perfis_doc = {a: v for a, v in perfis_doc.items() if a in autores_alvo}

if set(rankings) != autores_alvo:
    faltantes = sorted(autores_alvo - set(rankings))
    extras = sorted(set(rankings) - autores_alvo)
    raise ValueError(
        f"Rankings incompatíveis com o conjunto de teste. "
        f"Faltantes={len(faltantes)}, extras={len(extras)}"
    )

modelo_sbert = SentenceTransformer(SBERT_MODEL)

vocabulario = construir_vocabulario(
    rankings,
    gt_norm,
    perfis_doc,
    top_k_pred=TOP_K_MATCHING,
)

cache_emb = encodar_com_sbert(modelo_sbert, vocabulario)

print(f"Vocabulário semântico: {len(vocabulario):,} unidades")

## 6. Matching e métricas

In [ ]:
dados_por_autor = {}
matching_por_autor = {}
metricas_por_autor = {}

SIM_DIR.mkdir(parents=True, exist_ok=True)

for autor in autores_teste:
    dados = construir_matriz_similaridade(
        autor=autor,
        ranking=rankings[autor],
        gt_norm_autor=gt_norm.get(autor, {}),
        cache_emb=cache_emb,
        top_k=TOP_K_MATCHING,
    )

    if dados is None:
        continue

    matched_weights, matched_idx, matched_sims = matching_greedy_1_to_1(
        dados["sim"],
        dados["gold_weights"],
        theta=THRESHOLD,
    )

    salvar_npz(
        dados,
        matched_idx,
        matched_weights,
        matched_sims,
        SIM_DIR,
    )

    dados_por_autor[autor] = dados
    matching_por_autor[autor] = {
        "matched_weights": matched_weights,
        "matched_idx": matched_idx,
        "matched_sims": matched_sims,
    }

    docs_autor = perfis_doc.get(autor, {})
    metricas_por_autor[autor] = avaliar_autor(
        dados,
        matched_weights,
        docs_autor,
        docs_autor,
        cache_emb,
        theta_cov=THRESHOLD_COVERAGE,
    )

print(f"Autores avaliados: {len(metricas_por_autor)}")

## 7. Salvamento dos resultados

In [ ]:
salvar_csv_metricas(
    metricas_por_autor,
    METRICS_FILE,
    modelo="Qwen3.5-4B Zero-shot",
)

salvar_csv_avaliacoes_gerais(
    dados_por_autor,
    matching_por_autor,
    gt_original,
    MATCHING_FILE,
    modelo="Qwen3.5-4B Zero-shot",
    rank_max=20,
)

print(f"Métricas:             {METRICS_FILE}")
print(f"Avaliações detalhadas: {MATCHING_FILE}")
print(f"Matrizes de similaridade: {SIM_DIR}")

## 8. Resultado médio

In [ ]:
if metricas_por_autor:
    nomes = list(next(iter(metricas_por_autor.values())).keys())
    medias = {
        nome: float(np.mean([m[nome] for m in metricas_por_autor.values()]))
        for nome in nomes
    }
    for nome, valor in medias.items():
        print(f"{nome:18s}: {valor:.4f}")